In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
pdf_path = "../../records/pdfs/"
papers_path = "../../records/papers/"
self_notes_path = "../../records/self-notes/"
transcripts_path = "../../records/transcripts/"
generic_path = "../../records/generic/"

In [3]:
import os
import sys

# Add utilities directory to path so imports work
utilities_dir = os.path.join(os.getcwd(), 'src', 'utilities')
if utilities_dir not in sys.path:
    sys.path.insert(0, utilities_dir)

In [4]:
pdfs = os.listdir(pdf_path)

In [5]:
pdfs

['MACHINE LEARNING(R17A0534).pdf']

In [ ]:
from chunking.pdf_chunking import semantic_pdf_chunking
for pdf in pdfs:
    chunks_pdf = semantic_pdf_chunking(os.path.join(pdf_path, pdf))

In [ ]:
chunks_pdf

[{'chunk_id': 'c9109e6b-be32-4ad1-9713-1f4a20b3cb6b',
  'text': 'MACHINE LEARNING [R17A0534] LECTURE NOTES B.TECH IV YEAR – I SEM(R17) (2020-21) DEPARTMENT OF COMPUTER SCIENCE AND ENGINEERING MALLA REDDY COLLEGE OF ENGINEERING & TECHNOLOGY (Autonomous Institution – UGC, Govt. of India) Recognized under 2(f) and 12 (B) of UGC ACT 1956 (Affiliated to JNTUH, Hyderabad, Approved by AICTE - Accredited by NBA & NAAC – ‘A’ Grade - ISO 9001:2015 Certified) Maisammaguda, Dhulapally (Post Via. Hakimpet), Secunderabad – 500100, Telangana State, India IV Year B. CSE –II Sem L T/P/D C 4 1/- / - 3 (R17A0534) Machine Learning Objectives: \uf0b7 Acquire theoretical Knowledge on setting hypothesis for pattern recognition. \uf0b7 Apply suitable machine learning techniques for data handling and to gain knowledge from it. \uf0b7 Evaluate the performance of algorithms and to provide solution for various real world applications. UNIT I: Introduction to Machine Learning Introduction ,Components of Learning ,

In [7]:
papers = os.listdir(papers_path)

In [8]:
from chunking.paper_based_chunking import chunk_research_paper
for paper in papers:
    chunks_paper = chunk_research_paper(os.path.join(papers_path, paper))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4457.38it/s]


In [9]:
chunks_paper

[{'chunk_id': '998d7738-bfd7-4d72-9591-e2a085654740',
  'parent_chunk_id': 'section_0',
  'section': 'Unknown',
  'text': 'Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works. Attention Is All You Need Ashish Vaswani∗ Google Brain avaswani@google.com Noam Shazeer∗ Google Brain noam@google.com Niki Parmar∗ Google Research nikip@google.com Jakob Uszkoreit∗ Google Research usz@google.com Llion Jones∗ Google Research llion@google.com Aidan N. Gomez∗† University of Toronto aidan@cs.toronto.edu Łukasz Kaiser∗ Google Brain lukaszkaiser@google.com Illia Polosukhin∗‡ illia.polosukhin@gmail.com The dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder. The best performing models also connect the encoder and decoder through an attention mechanism. We propose a new simple network architecture, 

In [10]:
from database_handling.vector_db_handling import VectorStore

In [7]:
vector_store = VectorStore(

        persist_directory="./pkms_db",

        collection_name="knowledge_base"
    )


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 14283.79it/s]


In [8]:
vector_store.add_items(chunks_paper)

In [11]:
from entity_handlers.cluster_similar_chunks import consolidate_chunks

consolidated_chunks = consolidate_chunks(chunks_paper)

Batches: 100%|██████████| 1/1 [00:00<00:00,  3.53it/s]


In [12]:
consolidated_chunks

[{'cluster_id': np.int64(12),
  'text': 'Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works. Attention Is All You Need Ashish Vaswani∗ Google Brain avaswani@google.com Noam Shazeer∗ Google Brain noam@google.com Niki Parmar∗ Google Research nikip@google.com Jakob Uszkoreit∗ Google Research usz@google.com Llion Jones∗ Google Research llion@google.com Aidan N. Gomez∗† University of Toronto aidan@cs.toronto.edu Łukasz Kaiser∗ Google Brain lukaszkaiser@google.com Illia Polosukhin∗‡ illia.polosukhin@gmail.com The dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder. The best performing models also connect the encoder and decoder through an attention mechanism. We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence 

In [13]:
len(consolidated_chunks)

14

In [14]:
from entity_handlers.topic_summary_extractor import load_and_process_document

chunks = load_and_process_document(consolidated_chunks)

In [16]:
chunks

[{'cluster_id': np.int64(12),
  'text': 'Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works. Attention Is All You Need Ashish Vaswani∗ Google Brain avaswani@google.com Noam Shazeer∗ Google Brain noam@google.com Niki Parmar∗ Google Research nikip@google.com Jakob Uszkoreit∗ Google Research usz@google.com Llion Jones∗ Google Research llion@google.com Aidan N. Gomez∗† University of Toronto aidan@cs.toronto.edu Łukasz Kaiser∗ Google Brain lukaszkaiser@google.com Illia Polosukhin∗‡ illia.polosukhin@gmail.com The dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder. The best performing models also connect the encoder and decoder through an attention mechanism. We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence 

In [22]:
import pickle
with open('chunks.pkl', 'wb') as f:
    pickle.dump(chunks, f)

In [17]:
import database_handling.graph_db_handling

In [18]:
def get_graph_store():
    return database_handling.graph_db_handling.GraphStore()

In [19]:
graph_store = get_graph_store()

In [23]:
"""
Neo4j Aura Semantic Graph Store
--------------------------------
Architecture:

(Domain)
    ↓
(Topic)
    ↓
(Entity)

Features:
- Domain nodes
- Topic nodes
- Entity nodes
- Entity embeddings
- Entity descriptions
- Subdomain metadata
- Relationship edges
- Merge-ready entity structure

Install:
pip install neo4j
"""

from neo4j import GraphDatabase
import json


# =========================================================
# GRAPH STORE
# =========================================================
class GraphStore:

    def __init__(

        self,

        uri=None,

        username=None,

        password=None
    ):

        with open(
            "neo4j_settings.json",
            "r"
        ) as f:

            settings = json.load(f)

        self.driver = (
            GraphDatabase.driver(

                settings["NEO4J_URI"],

                auth=(

                    settings[
                        "NEO4J_USERNAME"
                    ],

                    settings[
                        "NEO4J_PASSWORD"
                    ]
                )
            )
        )

    # =====================================================
    # CLOSE
    # =====================================================
    def close(self):

        self.driver.close()

    # =====================================================
    # ADD DOMAIN
    # =====================================================
    def add_domain(

        self,

        tx,

        domain
    ):

        query = """
        MERGE (d:Domain {
            name: $domain
        })
        """

        tx.run(

            query,

            domain=domain
        )

    # =====================================================
    # ADD TOPIC
    # =====================================================
    def add_topic(

        self,

        tx,

        topic,

        domain
    ):

        query = """
        MERGE (t:Topic {
            name: $topic
        })

        MERGE (d:Domain {
            name: $domain
        })

        MERGE (t)-[:IN_DOMAIN]->(d)
        """

        tx.run(

            query,

            topic=topic,

            domain=domain
        )

    # =====================================================
    # ADD ENTITY
    # =====================================================
    def add_entity(

        self,

        tx,

        entity_name,

        topic,

        domain,

        subdomain,

        description,

        text,

        embedding,

        source_file
    ):

        query = """
        MERGE (e:Entity {
            name: $entity_name,
            subdomain: $subdomain
        })

        SET e.description = $description,
            e.text = $text,
            e.embedding = $embedding,
            e.source = $source_file

        MERGE (t:Topic {
            name: $topic
        })

        MERGE (e)-[:HAS_TOPIC]->(t)
        """

        tx.run(

            query,

            entity_name=entity_name,

            topic=topic,

            domain=domain,

            subdomain=subdomain,

            description=description,

            text=text,

            embedding=embedding,

            source_file=source_file
        )

    # =====================================================
    # ADD RELATIONSHIP
    # =====================================================
    def add_relationship(

        self,

        tx,

        source,

        target,

        relation,

        score=None
    ):

        query = f"""
        MERGE (a:Entity {{
            name: $source
        }})

        MERGE (b:Entity {{
            name: $target
        }})

        MERGE (a)-[r:{relation}]->(b)

        SET r.score = $score
        """

        tx.run(

            query,

            source=source,

            target=target,

            score=score
        )

        # =====================================================
    # ADD DOCUMENT
    # =====================================================
    def add_document(
        self,
        item
    ):

        # =================================================
        # META DETAILS
        # =================================================
        meta_details = item.get(
            "meta_details",
            {}
        )

        domain = meta_details.get(
            "domain",
            "Unknown"
        )

        subdomain = meta_details.get(
            "subdomain",
            "Unknown"
        )

        topics = meta_details.get(
            "topics",
            []
        )

        summary = meta_details.get(
            "summary",
            ""
        )

        # =================================================
        # DOCUMENT DETAILS
        # =================================================
        text = item.get(
            "text",
            ""
        )

        source_file = item.get(
            "source",
            ""
        )

        cluster_id = str(
            item.get(
                "cluster_id",
                ""
            )
        )

        entities = item.get(
            "entities",
            []
        )

        relationships = item.get(
            "relationships",
            []
        )

        # =================================================
        # SESSION
        # =================================================
        with self.driver.session() as session:

            # =============================================
            # DOMAIN
            # =============================================
            session.execute_write(

                self.add_domain,

                domain
            )

            # =============================================
            # TOPICS
            # =============================================
            for topic in topics:

                session.execute_write(

                    self.add_topic,

                    topic,

                    domain
                )

            # =============================================
            # ENTITIES
            # =============================================
            for entity in entities:

                # -----------------------------------------
                # Entity name
                # -----------------------------------------
                entity_name = str(
                    entity
                ).strip()

                if not entity_name:
                    continue

                # -----------------------------------------
                # Entity description
                # -----------------------------------------
                description = f"""
                Entity:
                {entity_name}

                Summary:
                {summary}

                Topics:
                {", ".join(topics)}

                Context:
                {text[:3000]}
                """

                # -----------------------------------------
                # Embedding placeholder
                # -----------------------------------------
                embedding = []

                # -----------------------------------------
                # Topic routing
                # -----------------------------------------
                topic = (

                    topics[0]

                    if topics

                    else "General"
                )

                # -----------------------------------------
                # Add entity
                # -----------------------------------------
                session.execute_write(

                    self.add_entity,

                    entity_name,

                    topic,

                    domain,

                    subdomain,

                    description,

                    text,

                    embedding,

                    source_file
                )

            # =============================================
            # RELATIONSHIPS
            # =============================================
            for rel in relationships:

                source = rel.get(
                    "source",
                    ""
                )

                target = rel.get(
                    "target",
                    ""
                )

                relation = rel.get(
                    "relation",
                    "RELATED_TO"
                )

                score = rel.get(
                    "score",
                    rel.get(
                        "weight",
                        1
                    )
                )

                if (
                    not source
                    or
                    not target
                ):
                    continue

                # -----------------------------------------
                # Neo4j-safe relation
                # -----------------------------------------
                relation = (

                    relation.upper()

                    .replace(" ", "_")

                    .replace("-", "_")
                )

                # -----------------------------------------
                # Add relationship
                # -----------------------------------------
                session.execute_write(

                    self.add_relationship,

                    source,

                    target,

                    relation,

                    score
                )

    # =====================================================
    # ADD MULTIPLE DOCUMENTS
    # =====================================================
    def add_documents(
        self,
        items
    ):

        for item in items:

            self.add_document(item)

    # =====================================================
    # QUERY
    # =====================================================
    def query(
        self,
        cypher_query
    ):

        with self.driver.session() as session:

            result = session.run(
                cypher_query
            )

            return [

                record.data()

                for record in result
                
            ]

In [24]:
graph_store = GraphStore()

In [25]:
graph_store.add_documents(chunks)

In [3]:
self_notes = os.listdir(self_notes_path)


In [4]:
from chunking.self_notes_chunking import chunk_markdown_note
for paper in self_notes:
    chunks_notes = chunk_markdown_note(os.path.join(self_notes_path, paper))

/Users/abhijit/Desktop/Stock_Project/PKMS_system/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9604.56it/s]


['#datascience', 'Tags']
['Machine Learning', 'a field', 'data', 'patterns', 'systems']
['Core idea', 'Data -> Pattern Learning -> Predictions / Decisions', 'ML systems', '``text', 'more data', 'text Data -> Pattern Learning -> Predictions / Decisions', 'they']
[]
['labeled data']
['* Spam detection', 'Examples', 'House price prediction', 'Linear Regression', 'Logistic Regression', 'Random Forest', 'Stock movement classification Algorithms', 'XGBoost * Neural Networks']
['hidden structure', 'unlabeled data']
['* Clustering Algorithms', '* DBSCAN * PCA * Autoencoders', 'Customer segmentation', 'Examples', 'K-Means', 'Topic modeling']
['Agent', 'penalties', 'rewards']
['Agent * Environment * Reward * Policy', 'Examples', 'Game AI * Trading agents Core Concepts', 'Robotics']
['text Data Collection ↓ Data Cleaning ↓ Feature Engineering ↓ Model Training ↓ Evaluation ↓ Deployment ↓ Monitoring']
['Input variables', 'the model']
['25 salary', 'Example', '```python age', 'python age = 25 salary

In [10]:
chunks_notes

NameError: name 'chunks_notes' is not defined

In [4]:
import fitz  # PyMuPDF

def extract_pdf_text(pdf_path):

    doc = fitz.open(pdf_path)

    full_text = []

    for page in doc:

        text = page.get_text("text")

        full_text.append(text)

    return "\n".join(full_text)

In [5]:
from entity_handlers.entity_extractor import extract_metadata

pdfs = os.listdir(pdf_path)

pdf_path = "../../records/pdfs/"

for pdf in pdfs:
    pdf_text = extract_pdf_text(os.path.join(pdf_path, pdf))
    metadata = extract_metadata(pdf_text)
    print(f"Metadata for {pdf}: {metadata}")


/Users/abhijit/Desktop/Stock_Project/PKMS_system/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10751.68it/s]


Metadata for MACHINE LEARNING(R17A0534).pdf: {'entities': [{'text': 'DEPARTMENT OF COMPUTER SCIENCE AND ENGINEERING MALLA REDDY COLLEGE OF ENGINEERING & TECHNOLOGY', 'normalized': 'department of computer science and engineering malla reddy college of engineering & technology', 'score': 0.9, 'source': 'spacy', 'label': 'ORG', 'tfidf_score': 14362, 'topic_rank_score': 1, 'final_score': 719.5}, {'text': 'Biological Neural Network Artificial Neural Network Dendrites Inputs Cell', 'normalized': 'biological neural network artificial neural network dendrites inputs cell', 'score': 0.9, 'source': 'spacy', 'label': 'ORG', 'tfidf_score': 11945, 'topic_rank_score': 1, 'final_score': 598.65}, {'text': 'The World Wide Web', 'normalized': 'the world wide web', 'score': 0.9, 'source': 'spacy', 'label': 'ORG', 'tfidf_score': 9677, 'topic_rank_score': 1, 'final_score': 485.25}, {'text': 'Linear Regression in Machine Learning Linear', 'normalized': 'linear regression in machine learning linear', 'score'

In [6]:
metadata

{'entities': [{'text': 'DEPARTMENT OF COMPUTER SCIENCE AND ENGINEERING MALLA REDDY COLLEGE OF ENGINEERING & TECHNOLOGY',
   'normalized': 'department of computer science and engineering malla reddy college of engineering & technology',
   'score': 0.9,
   'source': 'spacy',
   'label': 'ORG',
   'tfidf_score': 14362,
   'topic_rank_score': 1,
   'final_score': 719.5},
  {'text': 'Biological Neural Network Artificial Neural Network Dendrites Inputs Cell',
   'normalized': 'biological neural network artificial neural network dendrites inputs cell',
   'score': 0.9,
   'source': 'spacy',
   'label': 'ORG',
   'tfidf_score': 11945,
   'topic_rank_score': 1,
   'final_score': 598.65},
  {'text': 'The World Wide Web',
   'normalized': 'the world wide web',
   'score': 0.9,
   'source': 'spacy',
   'label': 'ORG',
   'tfidf_score': 9677,
   'topic_rank_score': 1,
   'final_score': 485.25},
  {'text': 'Linear Regression in Machine Learning Linear',
   'normalized': 'linear regression in machin

In [7]:
from entity_handlers.create_info_list import extract_knowledge

result = extract_knowledge(chunks_pdf)

In [8]:
result

[{'chunk_id': '0e435324-b0c8-40ad-9488-26d72d858997',
  'source': '../../records/pdfs/MACHINE LEARNING(R17A0534).pdf',
  'text': 'MACHINE LEARNING [R17A0534] LECTURE NOTES B.TECH IV YEAR – I SEM(R17) (2020-21) DEPARTMENT OF COMPUTER SCIENCE AND ENGINEERING MALLA REDDY COLLEGE OF ENGINEERING & TECHNOLOGY (Autonomous Institution – UGC, Govt. of India) Recognized under 2(f) and 12 (B) of UGC ACT 1956 (Affiliated to JNTUH, Hyderabad, Approved by AICTE - Accredited by NBA & NAAC – ‘A’ Grade - ISO 9001:2015 Certified) Maisammaguda, Dhulapally (Post Via. Hakimpet), Secunderabad – 500100, Telangana State, India IV Year B. CSE –II Sem L T/P/D C 4 1/- / - 3 (R17A0534) Machine Learning Objectives: \uf0b7 Acquire theoretical Knowledge on setting hypothesis for pattern recognition. \uf0b7 Apply suitable machine learning techniques for data handling and to gain knowledge from it. \uf0b7 Evaluate the performance of algorithms and to provide solution for various real world applications. UNIT I: Introd

In [7]:
from database_handling.vector_db_handling import VectorStore

In [8]:
vector_store = VectorStore(

        persist_directory="./pkms_db",

        collection_name="knowledge_base"
    )


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9825.63it/s]


In [9]:
vector_store.add_items(chunks_pdf)

In [10]:
vector_store.search("machine learning", top_k=3)

{'ids': [['c9109e6b-be32-4ad1-9713-1f4a20b3cb6b',
   '9678b0b8-5b7c-44f7-b9bd-f475235edaaa',
   'f21ed0f2-61a0-484a-8173-8f54ecc5f872']],
 'embeddings': None,
 'documents': [['MACHINE LEARNING [R17A0534] LECTURE NOTES B.TECH IV YEAR – I SEM(R17) (2020-21) DEPARTMENT OF COMPUTER SCIENCE AND ENGINEERING MALLA REDDY COLLEGE OF ENGINEERING & TECHNOLOGY (Autonomous Institution – UGC, Govt. of India) Recognized under 2(f) and 12 (B) of UGC ACT 1956 (Affiliated to JNTUH, Hyderabad, Approved by AICTE - Accredited by NBA & NAAC – ‘A’ Grade - ISO 9001:2015 Certified) Maisammaguda, Dhulapally (Post Via. Hakimpet), Secunderabad – 500100, Telangana State, India IV Year B. CSE –II Sem L T/P/D C 4 1/- / - 3 (R17A0534) Machine Learning Objectives: \uf0b7 Acquire theoretical Knowledge on setting hypothesis for pattern recognition. \uf0b7 Apply suitable machine learning techniques for data handling and to gain knowledge from it. \uf0b7 Evaluate the performance of algorithms and to provide solution for v

In [3]:
from entity_handlers.topic_summary_extractor import load_and_process_document

papers = os.listdir(papers_path)


docs = load_and_process_document(os.path.join(papers_path, papers[0]))

Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aidan@cs.toronto.edu
Łukasz K
{'domain': 'Unknown', 'subdomain': 'Unknown', 'topics': [], 'summary': ''}


In [4]:
docs

{'source': '../../records/papers/1706.03762v7.pdf',
 'domain': 'Unknown',
 'subdomain': 'Unknown',
 'topics': [],
 'summary': ''}